In [7]:
# esta rotina insere os registros novos na tabela anterioridades_desc
# atualiza o campo descricao: com a discussão de atividade inventiva mencionada no indeferimento
# atualiza o campo conclusao: com a conclusão do parecer
# esses campos são tomados literalmente do parecer de indeferimento sem qualquer processamento por IA
# eles serão usados por outras rotinas posteriores para serem enviadas a IA para processamento
# a análise toma como ponto de partida os pedidos na carga que sejam resultado de 12.2 apenas
# portanto deve ser rodada semanalmente na carga da nova revista

# select * from CEPIT_SISCAP.SISCAP_CARGA where numero in (select numero from CEPIT_SISCAP.SISCAP_arquivados WHERE despacho in ('12.2') and anulado=0)
# select * from carga where numero in (select numero from arquivados WHERE despacho in ('12.2') and anulado=0)
# select * from anterioridades_desc where 1
# select * from anterioridades where 1

import pandas as pd

In [8]:
# pip install mysql-connector-python
import mysql.connector
conexao = mysql.connector.connect(host='localhost',user='root',password='',database='producao')
cursor = conexao.cursor()

In [9]:
numero = "PI0808715"
# atualiza no localhost as tres tabelas: carga, anterioridades e anterioridades_desc
comando = f"SELECT * FROM arquivados WHERE numero='{numero}'"
cursor.execute(comando)
resultado = cursor.fetchall()
df = pd.DataFrame(resultado)
print(resultado)

[(1011532, '1.1', 'PI0808715', datetime.date(2011, 8, 9), 'dialp', 0, 0), (1475660, '1.3', 'PI0808715', datetime.date(2014, 8, 12), 'dialp', 0, 0), (1501853, '6.6', 'PI0808715', datetime.date(2014, 9, 16), 'dialp', 0, 0), (2788968, '7.1', 'PI0808715', datetime.date(2017, 4, 18), 'dialp', 0, 1), (2790019, '15.11', 'PI0808715', datetime.date(2017, 4, 18), 'dialp', 0, 0), (2956330, '9.2', 'PI0808715', datetime.date(2017, 9, 19), 'dialp', 0, 2), (3015281, '12.2', 'PI0808715', datetime.date(2017, 12, 19), 'dialp', 0, 0)]


In [10]:
# testa se a conexão com MySQL esta OK
import json
import requests

def conectar_siscap(url,return_json=False):
    headers = {
        "Accept": "application/json",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    response = requests.get(url,headers=headers,verify=False)
    if response.status_code == 200:
        if return_json:
            data = response.json()
            json_data = json.dumps(data, indent=4)
            return(json_data)
        else:
            return response.text
    else:
        return(f"Erro: {response.status_code}")

In [5]:
    
numero='PI0905487'
numero='102020022082'
numero='112016024777'
#numero='102012010730' # este teve indeferimneto técnico 9.2, não tem 'indeferimento'
# https://cientistaspatentes.com.br/apiphp/menu_api.php
# https://cientistaspatentes.com.br/apiphp/patents/query/?q={%22application_number%22:%22C10000061%22}
# https://cientistaspatentes.com.br/apiphp/patents/query/?q={%22mysql_query%22:%22%20*%20FROM%20arquivados%20where%20numero=%27PI0905487%27%22}
# https://cientistaspatentes.com.br/apiphp/patents/query/?q={%22mysql_query%22:%22%20*%20FROM%20pedido%20where%20numero=%27PI0905487%27%22}
# https://cientistaspatentes.com.br/apiphp/patents/query/?q={%22mysql_query%22:%22%20*%20FROM%20pedido%20where%20decisao=%27indeferimento%27%20and%20numero=%27PI0905487%27%22}
# https://cientistaspatentes.com.br/apiphp/patents/query/?q={"mysql_query":" * FROM pedido where decisao='indeferimento' and numero='PI0905487'"}
# https://cientistaspatentes.com.br/apiphp/patents/query/?q={"mysql_query":" * FROM carga where divisao='direp'"}
# https://cientistaspatentes.com.br/apiphp/patents/query/?q={%22mysql_query%22:%22%20*%20FROM%20carga%20where%20divisao=%27direp%27%22}

query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM pedido where decisao='indeferimento' and numero='{numero}'" + '"'
url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query}"
json_data = conectar_siscap(url,return_json=True)
    
#json_data='{"patents": [{"numero":"PI0905487","prioridade":"BR","instancia":"2 exame","decisao":"indeferimento","prioritario":"0","cc1":"4","anulado":"0","codigo":"1340921","rpi":"2020-12-29","divisao":"dicel","etapa":"2"}]}'
data = json.loads(json_data)
codigo = data["patents"][0]["codigo"]
divisao = data["patents"][0]["divisao"]
print(f"Código: {codigo}")
print(f"Divisão: {divisao}")

D:\Users\abrantes\InstallAnaconda\envs\python\lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'cientistaspatentes.com.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Código: 1590318
Divisão: dicel


In [ ]:
# Conecte na VPN e teste se captura parecer do siscap
url = f"https://siscap.inpi.gov.br/adm/pareceres/{divisao}/{numero}{codigo}.txt"
print(url)
texto_relatorio = conectar_siscap(url,return_json=False)
print(texto_relatorio)
caminho_do_arquivo=f"pareceres/{divisao}/{numero}{codigo}.txt"
with open(caminho_do_arquivo, 'w', encoding='utf-8') as arquivo:
    arquivo.write(texto_relatorio)

In [6]:
# testa se faz download de petição
## 112015014614
## petição 214
## 29409161954365011  RJ	11 - Pagamento Conciliado 870220079398	01/09/2022	214  
# NUMERO;PETICAO;NUMNOSSONUMERO;DATA_PETICAO;TIPO_PETICAO;FLAG_PEDEXAME;FLAG_IMAGEM;CD_IMAGEM;UPDATE_IMAGEM;CONCILIADO
# "112015014614";"WBRJ 870220079398";"29409161954365011";"01/09/22 00:00:00,000000000";"214";1;1;9201851;"2022-09-03 00:53:04";1
# {"patents": [{"numero":"112015014614","peticao":"WBRJ 870220079398","numnossonumero":"29409161954365011","data_peticao":"2022-09-01",
# "tipo_peticao":"214","flag_pedexame":"7","flag_imagem":"1","cd_imagem":"1","update_imagem":"0000-00-00 00:00:00","conciliado":"1"}]}

# numnossonumero = '29409161954365011'
url = "https://siscap.inpi.gov.br/adm/download.php?arquivo=29409161954365011.pdf&url=http://172.20.2.43:8080/medusa/imagens/868b32d2c3aca81ac70f51f04cd5da92970f0ae2c57bbec71e6d806ff35fa2ab/imagem"
url = "http://br00-aux.inpi.gov.br/webservice/retornaImagem.php?codigo=9201851"
arquivo_saida = "29409161954365011.pdf"

try:
    response = requests.get(url, stream=True, verify=False, timeout=30)

    if response.status_code == 200:
        with open(arquivo_saida, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
        print(f"Download concluído: {arquivo_saida}")
    else:
        print(f"Falha no download. HTTP {response.status_code}")

except requests.exceptions.RequestException as e:
    print(f"Erro na requisição: {e}")

Download concluído: 29409161954365011.pdf


In [7]:
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path='.env', override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
#print(openai_api_key)

In [ ]:
import mysql.connector
conexao = mysql.connector.connect(host='localhost',user='root',password='',database='producao')
cursor = conexao.cursor()

comando = f"select * from carga where numero<>'NUMERO' and numero not in (select numero from anterioridades_desc) and numero in (select numero from arquivados where despacho='12.2')"
comando = f"select * from carga where numero<>'NUMERO' and numero not in (select numero from anterioridades_desc)"
# teste se existe algum pedido na carga com 12.2 que ainda não tenha registro em anterioridades_desc:
# f"SELECT * FROM `carga` WHERE numero<>'NUMERO' and numero not in (select numero from anterioridades_desc) and numero in (select numero from arquivados where despacho='12.2');"
cursor.execute(comando)
resultado = cursor.fetchall()
df = pd.DataFrame(resultado)
#print(resultado)
lista = df.values.tolist()
if df.shape[1] > 1:
    lista = df.iloc[:, 0].tolist()
else:
    lista = []
lista.insert(0, 'numero')
print(lista)

In [ ]:
data["patents"] = lista
for i in range(1, len(data["patents"])):
    numero = data["patents"][i] 
    sql_resumo = f"INSERT IGNORE INTO anterioridades_desc (id,numero,descricao) VALUES (null, '{numero}', null);"
    print(sql_resumo)

In [ ]:
# ******************************************INSERT ANTERIORIDADES_DESC CAMPO DESCRICAO COM DISCUSSÃO DA ATIVIDADE INVENTIVA
# certifique-se de rodar as rotinas acima conectar_siscap e de esta a VPN ligada
# SELECT * FROM `carga` WHERE numero not in (select numero from anterioridades_desc)

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain.prompts.prompt import PromptTemplate
import re

def limpar_caracteres_especiais(texto):
    # Remove caracteres não imprimíveis
    texto = re.sub(r'[^\x20-\x7EÀ-ÿ]', '', texto)
    return texto
    
def format_as_single_paragraph(text):
    # Remove quebras de linha e espaços extras
    formatted_text = ' '.join(line.strip() for line in text.splitlines() if line.strip())
    return formatted_text
    
load_dotenv(dotenv_path='.env', override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
url_openai = "https://api.openai.com/v1/chat/completions"

query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM carga" + '"'
url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query}"
json_data = conectar_siscap(url,return_json=True)
data = json.loads(json_data)
#data["patents"] = ['numero','102015025149'] # lista de numeros especificos
data["patents"] = lista
with open("descricao.sql", "a", encoding="utf-8") as f:
    for i in range(1, len(data["patents"])):
        #if i==2: break
        #numero = data["patents"][i]["numero"]
        numero = data["patents"][i] # para ler a lista de numeros especificos
    
        #query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM pedido where decisao='indeferimento' and numero='{numero}'" + '"'
        query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM pedido where (decisao='indeferimento' or decisao='ciencia de parecer') and numero='{numero}'" + ' order by rpi desc"'
        url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query}"
        print(url)
        try:
            json_data = conectar_siscap(url,return_json=True)
            data1 = json.loads(json_data)
            codigo = data1["patents"][0]["codigo"]
            divisao = data1["patents"][0]["divisao"]    
            
            url = f"https://siscap.inpi.gov.br/adm/pareceres/{divisao}/{numero}{codigo}.txt"
            print(url)
            texto_relatorio = conectar_siscap(url,return_json=False)
            ##print(texto_relatorio)
        
            url = "https://api.openai.com/v1/chat/completions"
            query = f"Selecione no texto seguinte apenas a parte que fala das diferenças com os documentos do estado da técnica e discute a atividade inventiva. Se não encontrar nenhuma discussão mostre apenas a conclusão final {texto_relatorio}"
            data_json = {
                "model": "gpt-5-mini",  # Use o modelo desejado, como 'gpt-4'
                "messages": [
                    {"role": "user", "content": query}
                ]
            }
            headers = {
                "Authorization": f"Bearer {openai_api_key}",
                "Content-Type": "application/json"
            }
            response = requests.post(url, headers=headers, json=data_json, verify=False)
            if response.status_code == 200:
                resposta = response.json()
                resumo = resposta['choices'][0]['message']['content']
                resumo = resumo.replace("'","")
                resumo = resumo.replace('"',"")
                resumo = resumo.replace('---',"")
                resumo = resumo.replace('',',')
                resumo = resumo.replace('',',')
                resumo = resumo.replace('',',')
                resumo = resumo.replace('',',')
                resumo = resumo.replace('O trecho que fala sobre as diferenças com os documentos D1, D2, D3 e D4 e discute a atividade inventiva é o seguinte:','')
                resumo = resumo.replace('O trecho que fala das diferenças com os documentos D1, D2, D3 e D4 e discute a atividade inventiva é o seguinte:','')
                resumo = resumo.replace('A parte do texto que fala das diferenças com os documentos D1, D2, D3 e D4 e discute a atividade inventiva é a seguinte:','')
                resumo = resumo.replace('A parte do texto que fala sobre as diferenças com os documentos D1, D2, D3 e D4 e discute a atividade inventiva é a seguinte:','')
                resumo = resumo.replace('A parte que fala sobre as diferenças com os documentos D1, D2, D3 e D4 e discute a atividade inventiva é a seguinte:','')
                resumo = resumo.replace('A parte que fala das diferenças com os documentos D1, D2, D3 e D4 e discute a atividade inventiva é a seguinte:','')
                resumo = resumo.replace('Segue a parte do texto que fala sobre as diferenças com os documentos D1, D2, D3 e D4 e discute a atividade inventiva','')
                resumo = format_as_single_paragraph(resumo)
                resumo = limpar_caracteres_especiais(resumo)
                sql_resumo = f"INSERT IGNORE INTO anterioridades_desc (id,numero,descricao) VALUES (null, '{numero}','{resumo}');"
                print(sql_resumo)
                f.write(sql_resumo + "\n")
            else:
                print(f"Erro {response.status_code}: {response.text}")
                
            #if i == 2:
                #break
        except Exception as e:
            print(f"Não achei parecer de indeferimento {numero} {e}")

In [10]:
# aplique os INSERTs obtidos na célula anterior na tabela local do computador

import mysql.connector
conexao = mysql.connector.connect(host='localhost',user='root',password='',database='producao')
cursor = conexao.cursor()

comando = f"select * from anterioridades_desc where conclusao='';"
cursor.execute(comando)
resultado = cursor.fetchall()
df = pd.DataFrame(resultado)
#print(resultado)
lista = df.values.tolist()
lista = df.iloc[:, 1].tolist()
lista.insert(0, 'numero')
print(lista)

['numero', '112020025305', '102017014626', '112019011142', 'PI0703369', 'PI1105114', '112014031679', 'PI0207878', 'PI0208569', 'PI0210514', '102012009754', '102014018446', '102014031393', '112013024824', '112013026802', '112014031439', '112015003693', '112015006953', '112015006954', '112015032570', '112016002831', '122019014983', '122019015499', '122019015505', '122021015471', '202012011047', '202014031373', 'MU9101438', 'MU9102962', 'PI1106243', '102012004682', '102012009184', '112016001884', '122019015504', '122019015516', '202013018221', '102012010955', '102013005581', '102014025902', '102018011384', '112012030718', '202014000433', '202020012873', 'PI0416669', '102012010956', '112013002786', '112014000985', 'PI0922619', '102014027536', '112016000345', '112022018674', '102015027008', '112013013718', '122020024330', '112018069853', 'PI1106082', '122023000426', '102022023667', '112017009599', '112021009318', '102014029382', '112016011341', '112020024219', '112013017685', '112023005463'

In [ ]:
query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM carga" + '"'
url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query}"
json_data = conectar_siscap(url,return_json=True)
dados = json.loads(json_data)
lista = [p['numero'] for p in dados['patents'] if p['numero'] != 'NUMERO']
print(lista)

In [ ]:
## UPDATE anterioridades_desc campo conclusao, esta rotina não usa LLM
# certifique-se de rodar a rotina acima conectar_siscap e de esta a VPN ligada
# SELECT * FROM `carga` WHERE numero not in (select numero from anterioridades_desc)
# UPDATE anterioridades_desc SET descricao = REPLACE(descricao, 'D1, D2, D3 e D4', 'de anterioridade') WHERE descricao LIKE '%D1, D2, D3 e D4%';
# primeiro processe os pedidos com indeferimento tecnico

import os, re
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain.prompts.prompt import PromptTemplate

def limpar_caracteres_especiais(texto):
    # Remove caracteres não imprimíveis
    texto = re.sub(r'[^\x20-\x7EÀ-ÿ]', '', texto)
    return texto
 
def remove_page_and_pid(text: str) -> str:
    # remove ocorrência "Página <n>" possivelmente seguida por código PI...
    text = re.sub(r'(?i)\bPágina\s*\d+(?:\s*(?:de|/)\s*\d+)?', '', text)
    # remove ocorrência "Página <n> PI..." (se sobrar algum resíduo)
    text = re.sub(r'(?i)\bPágina\s*\d+(?:\s+PI\d+(?:-\d+)?)?', '', text)
    # remove códigos PI isolados como "PI1009860-7" ou "PI 1009860-7"
    text = re.sub(r'\bPI\s*\d+(?:-\d+)?\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\bBR\s*\d+(?:-\d+)?\b', '', text, flags=re.IGNORECASE)
    # remove sequências de espaços em branco extras
    text = re.sub(r'\s{2,}', ' ', text)
    # remove espaços antes de pontuação
    text = re.sub(r'\s+([,.;:!?])', r'\1', text)
    # remover traços/lists soltos do tipo " - " ou " — " que ficaram isolados
    text = re.sub(r'\s*[-–—]\s*(?=[^\w-])', ' ', text)
    # remover traços isolados no começo ou fim de linhas/frases
    text = re.sub(r'^[\s\-–—]+', '', text)
    text = re.sub(r'[\s\-–—]+$', '', text)
    # remover espaços duplos novamente e aparar
    text = re.sub(r'\s{2,}', ' ', text).strip()
    text = text.replace("..",".")
    text = text.replace(". .",".")
    text = text.replace("Art .8","Art 8")
    return text

def format_as_single_paragraph(text):
    # Remove quebras de linha e espaços extras
    formatted_text = ' '.join(line.strip() for line in text.splitlines() if line.strip())
    return formatted_text
    
with open("descricao.sql", "a", encoding="utf-8") as f:
    for i in range(1, len(lista)):
        #numero = data["patents"][i]["numero"]
        numero = lista[i] # para ler a lista de numeros especificos
    
        #query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM pedido where decisao='indeferimento' and numero='{numero}'" + '"'
        query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM pedido where (decisao='indeferimento' or decisao='ciencia de parecer') and numero='{numero}'" + ' order by rpi desc"'
        url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query}"
        print(url)
        try:
            json_data = conectar_siscap(url,return_json=True)
            data1 = json.loads(json_data)
            codigo = data1["patents"][0]["codigo"]
            divisao = data1["patents"][0]["divisao"]    
            
            url = f"https://siscap.inpi.gov.br/adm/pareceres/{divisao}/{numero}{codigo}.txt"
            print(url)
            texto_relatorio = conectar_siscap(url,return_json=False)
            
            #match = re.search(r"(CONCLUS[aã]O\s*[\r\n]+.*?)(?:Rio de Janeiro|$)", texto_relatorio, flags=re.S | re.I)
            match = re.search(
                r"^\s*[-–—]?\s*CONCLUS[aã]O\s*:?\s*$\s*(.*?)(?:^\s*Rio de Janeiro|\Z)",
                texto_relatorio,
                flags=re.S | re.I | re.M
            )
    
            if match:
                conclusao = re.sub(r"\s+", " ", match.group(1)).strip()
                print("Conclusão extraída:\n")
                conclusao = conclusao.replace("Conclusão", "")
                conclusao = conclusao.replace("CONCLUSÃO", "")
                conclusao = conclusao.replace("Assim sendo,", "")
                conclusao = conclusao.replace("","")
                conclusao = conclusao.replace("","")
                conclusao = conclusao.replace("","")
                conclusao = conclusao.replace("O depositante deve se manifestar quanto ao contido neste parecer em até 90 (noventa) dias, a partir da data de publicação na RPI, de acordo com o Art. 36 da LPI","")
                conclusao = conclusao.replace("Publique-se a ciência de parecer (7.1).","")
                conclusao = conclusao.split("De acordo com o Art. 212")[0].strip()
                conclusao = conclusao[0].upper() + conclusao[1:]
                conclusao = conclusao.replace("'","")
                conclusao = remove_page_and_pid(conclusao)
                conclusao = conclusao.replace("Código:5975ce1e23737deeb22c88e902a79153versão1.3 19/04/12","");
                conclusao = limpar_caracteres_especiais(conclusao)
                sql = f"update anterioridades_desc set conclusao='{conclusao}' where numero='{numero}';"
                print(sql)
                f.write(sql + "\n")
            elif "Assim sendo, de acordo com o Art. 37" in texto_relatorio:
                pos = texto_relatorio.find("Assim sendo, de acordo com o Art. 37")
                conclusao = texto_relatorio[pos:]
                conclusao = conclusao.strip()
                print("Conclusão extraída:\n")
                conclusao = conclusao.replace("Conclusão", "")
                conclusao = conclusao.replace("CONCLUSÃO", "")
                conclusao = conclusao.replace("Assim sendo,", "")
                conclusao = conclusao.replace("","")
                conclusao = conclusao.replace("","")
                conclusao = conclusao.replace("","")
                conclusao = conclusao.replace("O depositante deve se manifestar quanto ao contido neste parecer em até 90 (noventa) dias, a partir da data de publicação na RPI, de acordo com o Art. 36 da LPI","")
                conclusao = conclusao.replace("Publique-se a ciência de parecer (7.1).","")
                conclusao = conclusao.split("De acordo com o Art. 212")[0].strip()
                conclusao = conclusao[0].upper() + conclusao[1:]
                conclusao = conclusao.replace("'","")
                conclusao = remove_page_and_pid(conclusao)
                conclusao = conclusao.replace("Código:5975ce1e23737deeb22c88e902a79153versão1.3 19/04/12","");
                conclusao = limpar_caracteres_especiais(conclusao)
                sql = f"update anterioridades_desc set conclusao='{conclusao}' where numero='{numero}';"
                print(sql)
                f.write(sql + "\n")
            else:
                print(f"Trecho não encontrado {numero}.")
        
        except Exception as e:
            print(f"Não achei parecer de indeferimento {numero} {e}")

In [13]:
# repita a rotina acima com estes números com indeferimento adinistrativo
import mysql.connector
conexao = mysql.connector.connect(host='localhost',user='root',password='',database='producao')
cursor = conexao.cursor()

#comando = "SELECT * FROM `anterioridades_desc` WHERE numero in (select numero from arquivados where despacho='12.2') and numero in (select numero from pedido where decisao='9.2') and conclusao='';"
comando = "SELECT * FROM `carga` WHERE numero not in (select numero from anterioridades_desc) and numero in (select numero from arquivados where despacho='12.2') and numero in (select numero from pedido where decisao='9.2');"
cursor.execute(comando)
resultado = cursor.fetchall()
df = pd.DataFrame(resultado)

if df.empty:
    print("Nenhum registro encontrado.")
    lista = ['numero']   # cria lista mínima para não quebrar a rotina
else:
    #print(resultado)
    lista = df.values.tolist()
    lista = df.iloc[:, 0].tolist()
    lista.insert(0, 'numero')
    print(lista)

Nenhum registro encontrado.


In [17]:
# Processe os pedidos com indeferimento administrativo decisao = 9.2 e faça os INSERT em anterioridades_desc

import os

def extrair_indefiro_sem_publique(texto: str) -> str:
    # Captura do "Portanto, INDEFIRO" até antes de "Publique-se"
    padrao = r"(Portanto,?\s*INDEFIRO.*?)(?=Publique-se)"
    match = re.search(padrao, texto, flags=re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip()
    return ""
    
#lista = ['','102022014500']
with open("descricao.sql", "a", encoding="utf-8") as f:
    for i in range(1, len(lista)):
        #numero = data["patents"][i]["numero"]
        numero = lista[i] # para ler a lista de numeros especificos
    
        #query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM pedido where decisao='indeferimento' and numero='{numero}'" + '"'
        query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM pedido where (decisao='9.2') and numero='{numero}'" + ' order by rpi desc"'
        url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query}"
        print(url)
        try:
            json_data = conectar_siscap(url,return_json=True)
            data1 = json.loads(json_data)
            codigo = data1["patents"][0]["codigo"]
            divisao = data1["patents"][0]["divisao"]    
            
            url = f"https://siscap.inpi.gov.br/adm/pareceres/{divisao}/{numero}{codigo}.txt"
            print(url)
            texto_relatorio = conectar_siscap(url,return_json=False)
            
            match = re.search(r"(O depositante deixou.*?)(?=Rio de Janeiro)", texto_relatorio, flags=re.S)
    
            if match:
                conclusao = re.sub(r"\s+", " ", match.group(1)).strip()
                print("Conclusão:\n")
                conclusao = conclusao[0].upper() + conclusao[1:]
                conclusao = conclusao.replace("'","")
                conclusao = extrair_indefiro_sem_publique(conclusao)
                conclusao = limpar_caracteres_especiais(conclusao)
                sql = f"INSERT IGNORE INTO anterioridades_desc (id,numero,descricao,conclusao) VALUES (null, '{numero}','','{conclusao}');"
                #sql = f"UPDATE anterioridades_desc SET conclusao='{conclusao}' WHERE numero='{numero}';"
                print(sql)
                f.write(sql + "\n")
            else:
                print(f"Trecho não encontrado {numero}.")
                
        
        except Exception as e:
            print(f"Não achei parecer de indeferimento {numero} {e}")

In [ ]:
import mysql.connector
conexao = mysql.connector.connect(host='localhost',user='root',password='',database='producao')
cursor = conexao.cursor()

# select * from CEPIT_SISCAP.SISCAP_CARGA where numero in (select numero from CEPIT_SISCAP.SISCAP_arquivados WHERE despacho='12.2' and anulado=0)
# agora faça o import com o CSV gerado
# verifique se anterioridades tem duplicatas:
# https://cientistaspatentes.com.br/central/control.php?action=172
# teste se existem duplicatas
# SELECT numero, codigo, COUNT(*) AS total FROM anterioridades  GROUP BY numero, codigo HAVING COUNT(*) > 1;

comando = f"select * from carga where numero not in (select numero from anterioridades) and numero in (select numero from arquivados where despacho='12.2')"
#comando = f"select * from anterioridades_desc where numero not in (select numero from anterioridades) and numero in (select numero from arquivados where despacho='12.2')"
cursor.execute(comando)
resultado = cursor.fetchall()
df = pd.DataFrame(resultado)
#print(resultado)
lista = df.values.tolist()
lista = df.iloc[:, 0].tolist()
#lista = ['102021010219']
print(lista)

# muitos destes pedidos são dupla proteção, artigo 32 e artigo 10 que não tem mesmo anterioridades citadas

In [ ]:
# ******************************************
# gera lista de docs para cada um dos pedidos na carga da tabela anterioridades. Não usa LLM
# rodar rotina abaixo com VPN ligada certifique-se que siscap.inpi.gov.br funcionando

import re
from datetime import datetime
import json
import requests  # Supondo que conectar_siscap use requests

def converter_data(data):
    if not data:
        return None
    # Remove qualquer coisa que não seja número ou /
    data = re.sub(r'[^0-9/]', '', data)
    formatos = ['%d/%m/%y', '%d/%m/%Y']
    for fmt in formatos:
        try:
            return datetime.strptime(data, fmt).strftime('%Y-%m-%d')
        except ValueError:
            pass  # tenta o próximo formato
    return None

def montar_url_parecer(numero):
    query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM pedido where (decisao='indeferimento' or decisao='ciencia de parecer') and numero='{numero}' order by rpi desc" + '"'
    url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query}"
    json_data = conectar_siscap(url,return_json=True)
    if json_data is not None:
        data = json.loads(json_data)
        if not json_data or "patents" not in json_data or len(data["patents"]) == 0:
            return None
        else:
            data = json.loads(json_data)
            codigo = data["patents"][0]["codigo"]
            divisao = data["patents"][0]["divisao"]
            # print(f"Código: {codigo}")
            # print(f"Divisão: {divisao}")
            url = f"https://siscap.inpi.gov.br/adm/pareceres/{divisao}/{numero}{codigo}.txt"
            return url
    else:
        return None

saida = ''
url = ''
total = 0
query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM carga WHERE divisao='direp'" + '"'
url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query}"
json_data = conectar_siscap(url,return_json=True)
data = json.loads(json_data)
numbers = [patent['numero'] for patent in data['patents']]
#numbers = ['102019009508']
#lista = ['202014005715']
numbers = lista

with open("descricao.sql", "a", encoding="utf-8") as f:
    for numero in numbers:
        total = total + 1
        if (total>1000):
            break
        url = montar_url_parecer(numero)
        if url is not None:
            print(numero)
            print(url)
            
            texto_relatorio = conectar_siscap(url,return_json=False)
            #print(texto_relatorio)
            if texto_relatorio is not None:
                #caminho_do_arquivo='document.txt'
                #with open(caminho_do_arquivo, 'w', encoding='utf-8') as arquivo:
                #    arquivo.write(texto_relatorio)
    
                pattern = r"(D\d+)\s+((?:[A-Z]{2,3})\s*\d{4,10}(?:-\d[A-Z]?)?)\s+.*?(\d{2}[\/\.]\d{2}[\/\.]\d{4})"
                pattern = r"(D\d+)\s+([A-Z]{2,3}\s*\d[\d\.,]*?)\s+(\d{2}[\/\.]\d{2}[\/\.]\d{4})"
                pattern = r"(D\d+)\s+([A-Z]{2,3}\d+\s*[A-Z]?\d?)\s+(\d{2}[\/\.]\d{2}[\/\.]\d{4})"
                pattern = r"(D\d+)\s+([A-Z]{2,3}\s*\d+(?:-\d+)?)\s+(\d{2}[\/\.]\d{2}[\/\.]\d{4})"
                pattern =   r"""
                            (D\d+)                                      # Código D1, D2...
                            \s+
                            (
                                [A-Z]{2,3}                               # País (PI, BR, US, WO, EP...)
                                \s*
                                \d{4,}                                   # Número principal (mín 4 dígitos)
                                (?:[\/\-]\d+)?                           # Parte opcional tipo 2019/123456 ou -2
                                (?:\s*[A-Z]\d)?                          # Sufixo opcional tipo A1, B1
                            )
                            [\s\S]*?
                            (\d{2}[\/\.]\d{2}[\/\.]\d{4})                 # Data
                            """
    
                #matches = re.findall(pattern, texto_relatorio)
                matches = re.findall(pattern, texto_relatorio, re.VERBOSE | re.IGNORECASE)
                for match in matches:
                    #codigo, documento, tipo, data = match
                    codigo = match[0]
                    documento = match[1]
                    doc = " ".join(documento.split())
                    doc = doc.replace(" ", "")
                    doc = doc.replace("/", "")
                    doc = doc.replace(".", "")
                    doc = doc.replace(",", "")
                    doc = doc.replace(";", "")
                    doc = doc.split("-")[0]
                    data = match[-1]
                    data = converter_data(data)
                    #data = datetime.strptime(data, '%d/%m/%Y') # 11/07/2013
                    #data = data.strftime('%Y-%m-%d')
                    ##print(f"{codigo}: {doc}, Data de Publicação = {data}")
                    sql = f"INSERT IGNORE INTO anterioridades (numero, codigo, doc, data) VALUES ('{numero}','{codigo}','{doc}','{data}');"
                    print(sql)
                    saida = saida + f"INSERT IGNORE INTO anterioridades (numero, codigo, doc, data) VALUES ('{numero}','{codigo}','{doc}','{data}');\n"
                    f.write(sql + "\n")

print(saida)
# teste regex https://regex101.com/

In [46]:
import mysql.connector
conexao = mysql.connector.connect(host='localhost',user='root',password='',database='producao')
cursor = conexao.cursor()

comando = f"select * from anterioridades_desc where razoes='' AND numero in (select numero from carga) and numero in (select numero from arquivados where despacho='12.2')"
cursor.execute(comando)
resultado = cursor.fetchall()
df = pd.DataFrame(resultado)
#print(resultado)
lista = df.values.tolist()
lista1 = lista
lista = df.iloc[:, 1].tolist()
lista.insert(0, 'numero')
lista = ['numero','102021010219']
print(lista)

['numero', '102021010219']


In [ ]:
# ******************************************ATUALIZA ANTERIORIDADES_DESC CAMPO RAZOES COM RAZOES INDEFERIMENTO
# certifique-se de rodar a rotina acima conectar_siscap e de esta a VPN ligada
# SELECT * FROM `carga` WHERE numero not in (select numero from anterioridades_desc)
# esta rotina usa o openai
# https://platform.openai.com/settings/organization/billing/overview

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain.prompts.prompt import PromptTemplate

def format_as_single_paragraph(text):
    # Remove quebras de linha e espaços extras
    formatted_text = ' '.join(line.strip() for line in text.splitlines() if line.strip())
    return formatted_text
    
load_dotenv(dotenv_path='.env')
openai_api_key = os.getenv("OPENAI_API_KEY")
url_openai = "https://api.openai.com/v1/chat/completions"

query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM carga" + '"'
url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query}"
json_data = conectar_siscap(url,return_json=True)
data = json.loads(json_data)
#data["patents"] = ['102014022400','102014025472','102014027536','102018000306','102019009508']
data["patents"] = lista
with open("descricao.sql", "a", encoding="utf-8") as f:
    for i in range(1, len(data["patents"])):
        #if i==5: break
        numero = data["patents"][i]
    
        comando = f"SELECT codigo, doc FROM anterioridades WHERE numero = '{numero}'"
        print(comando)
        cursor.execute(comando)
        resultado = cursor.fetchall()
        documentos = ", ".join(f"{codigo} {doc}" for codigo, doc in resultado)
        #print(documentos)
        
        #query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM pedido where decisao='indeferimento' and numero='{numero}'" + '"'
        query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM anterioridades_desc where numero='{numero}'" + ' "'
        url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query}"
        print(url)
        try:
            json_data = conectar_siscap(url,return_json=True)
            data1 = json.loads(json_data)
            descricao = data1['patents'][0]['descricao']
            conclusao = data1['patents'][0]['conclusao']
       
            url = "https://api.openai.com/v1/chat/completions"
            query = f"""Voce é um assistente administrativo que deve identificar os artigos da LPI que fundamentam o indeferimento de um pedido de patente 
            tendo em vista a conclusão do parecer: ###{conclusao}###. No final cite apenas os artigos mencionados na conclusão do parecer e 
            escreva no seguinte formato: 'o artigo 25 por falta de clareza, o artigo 6° por dupla proteção, o artigo 32 por acréscimo de matéria, 
            a combinação dos artigos 8° e 11 por falta de novidade, 
            a combinação dos artigos 8° e 13 por falta de atividade inventiva (identifique cada documento mencionado na discussão do parecer 
            ###{descricao}### conforme na busca {documentos}. Note que nem todos os documentos da busca fundamentam a falta de atividade inventiva), 
            a combinação dos artigos 9° e 14 por falta de ato inventivo (identifique cada documento mencionado na discussão do parecer 
            conforme na busca {documentos}. Note que nem todos os documentos da busca fundamentam a falta de ato inventivo),
            o artigo 10 inciso III por não ser considerado invenção, o artigo 18 inciso III por não ser considerado matéria patenteável, 
            o artigo 24 por insuficiência descritiva, o artigo 15 por falta de aplicação industrial'. 
            Não faça referência ao artigo 37. Exiba como resposta simplesmente o texto final no formato solicitado
            """
    
            print(query)
            data_json = {
                "model": "gpt-5-mini",  # Use o modelo desejado, como 'gpt-4'
                "messages": [
                    {"role": "user", "content": query}
                ]
            }
            headers = {
                "Authorization": f"Bearer {openai_api_key}",
                "Content-Type": "application/json"
            }
            response = requests.post(url, headers=headers, json=data_json, verify=False)
            if response.status_code == 200:
                resposta = response.json()
                resumo = resposta['choices'][0]['message']['content']
                resumo = resumo.replace("'","")
                resumo = resumo.replace('"',"")
                resumo = resumo.replace('---',"")
                resumo = resumo.replace('',',')
                resumo = resumo.replace('',',')
                resumo = resumo.replace('',',')
                resumo = resumo.replace('',',')
                resumo = format_as_single_paragraph(resumo)
                print(resumo)
                sql_razoes = f"UPDATE anterioridades_desc set razoes='{resumo}' WHERE numero='{numero}';"
                print(sql_razoes)
                f.write(sql_razoes + "\n")
            else:
                print(f"Erro {response.status_code}: {response.text}")
                
        except Exception as e:
            print(f"Não achei parecer de indeferimento {numero} {e}")

In [17]:
# confira o resultado especialmente artigo 9 que costuma colocar falta de atividade inventiva, artigo 6 por dupla proteção , artigo 22 cita documento
# verifique em control.php?action

import mysql.connector
conexao = mysql.connector.connect(host='localhost',user='root',password='',database='producao')
cursor = conexao.cursor()

comando = f"select * from anterioridades_desc where incoerencia='' and numero in (select numero from carga);"
cursor.execute(comando)
resultado = cursor.fetchall()
df = pd.DataFrame(resultado)
#print(resultado)
lista = df.values.tolist()
lista1 = lista
lista = df.iloc[:, 1].tolist()
lista.insert(0, 'numero')
#print(lista)
lista = [x for x in lista if x != "numero"]
json_data = {"patents": [{"numero": item} for item in lista]}
print(json_data)

{'patents': [{'numero': 'PI1010415'}, {'numero': '102015025507'}, {'numero': 'PI1014081'}, {'numero': '122014023771'}, {'numero': 'PI0708406'}, {'numero': '122020017894'}, {'numero': '102016030353'}, {'numero': '112013009746'}, {'numero': '112013027356'}, {'numero': '112014026306'}, {'numero': '112020003854'}, {'numero': '112020014633'}, {'numero': '202012026251'}, {'numero': '202012030729'}, {'numero': '202013003796'}, {'numero': 'MU9102962'}, {'numero': 'PI0900922'}, {'numero': '202014012390'}, {'numero': '102015017470'}, {'numero': '112013029300'}, {'numero': '112016006522'}, {'numero': '112016006534'}, {'numero': '202015030495'}, {'numero': '102012023815'}, {'numero': '102014002008'}, {'numero': '102016024247'}, {'numero': '112013028037'}, {'numero': '112016005819'}, {'numero': '112016013344'}, {'numero': '122014030028'}, {'numero': '122014030029'}, {'numero': 'MU9001502'}, {'numero': '102014022400'}, {'numero': '112015030093'}, {'numero': '202015005230'}, {'numero': '202014006525'

In [ ]:
##################################
## UPDATE anterioridades_desc campo incoerencia
# https://platform.openai.com/settings/organization/billing/overview
# habilita modelo gpt-5-mini
# https://platform.openai.com/settings/organization/general  Limits  e selecione os modelos
# GPT-5.2 custa US$ 1,750 / 1 milhão de tokens https://openai.com/pt-BR/api/pricing/
# GPT-5-mini custa US$ 0,250 / 1 milhão de tokens

import os
from dotenv import load_dotenv

load_dotenv(dotenv_path='.env', override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")

query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM carga" + '"'
url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query}"
print(url)
json_data = conectar_siscap(url,return_json=True)
    
#json_data='{"patents": [{"numero":"PI0905487","prioridade":"BR","instancia":"2 exame","decisao":"indeferimento","prioritario":"0","cc1":"4","anulado":"0","codigo":"1340921","rpi":"2020-12-29","divisao":"dicel","etapa":"2"}]}'
#json_data='{"patents": [{"numero":"122019017408"}]}'
json_data = {"patents": [{"numero": item} for item in lista]}
json_data = json.dumps(json_data, indent=4, ensure_ascii=False)
#print(json_data)
data = json.loads(json_data)

with open("descricao.sql", "a", encoding="utf-8") as f:
    for patent in data.get("patents", []):
        numero = patent.get("numero")
        if numero != 'NUMERO' and divisao != 'sanot':
            query_pedido = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM pedido where decisao='indeferimento' and numero='{numero}'" + '"'
            url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query_pedido}"
            #print(url)
            try:
                json_data = conectar_siscap(url,return_json=True)
            except:
                print(f"conexão inválida, pulando (indeferimento {numero} não encontrado)...")
                continue
                
            if not json_data:
                print("json_data vazio, pulando...")
                continue
            try:
                data_pedido = json.loads(json_data)
            except json.JSONDecodeError:
                print("JSON inválido, pulando...")
                continue
    
            if not data_pedido["patents"]:
                print(f"Sem registros para {numero}, pulando...")
                continue
        
            #data_pedido = json.loads(json_data)
            codigo = data_pedido["patents"][0]["codigo"]
            divisao = data_pedido["patents"][0]["divisao"]
        
            url = f"https://siscap.inpi.gov.br/adm/pareceres/{divisao}/{numero}{codigo}.txt"
            #print(url)
        
            texto_relatorio = conectar_siscap(url, return_json=False)
            #print(texto_relatorio)
            url = "https://api.openai.com/v1/chat/completions"
            query = f"""Neste parecer verifique se existência de incoerência interna entre as conclusões e o restante do parecer. 
            Ignore os X nos quadros 2 e 3 e considere apenas a discussão que se segue nestes quadros (se houver).
            Quando apenas uma ou mais reivindicação não tem novidade ou atividade inventiva, é justificável indeferir o pedido por falta de novidade 
            ou atividade inventiva, respectivamente. Escreva a saída numa forma corrida, sem bullets, parágrafos, travessões, nem pula linha.
            Apresente uma resposta curta e objetiva. Relatório: {texto_relatorio}"""
            data_json = {
                "model": "gpt-5-mini",  # Use o modelo desejado, como 'gpt-5.2' ou 'gpt-4o-mini' ou 'gpt-5-mini'
                "messages": [
                    {"role": "user", "content": query}
                ]
            }
            headers = {
                "Authorization": f"Bearer {openai_api_key}",
                "Content-Type": "application/json"
            }
            #response = requests.post(url, headers=headers, json=data_json, verify=False)
            #if response.status_code == 200:
            #    resposta = response.json()
            #    resumo = resposta['choices'][0]['message']['content']
            #    sql_resumo = f"UPDATE anterioridades_desc set incoerencia='{resumo}' WHERE numero='{numero}';"
            #    print(sql_resumo)
            #else:
            #    print(f"Não consegui conexão {numero}")
    
            response = requests.post(url, headers=headers, json=data_json, verify=False)
            
            if response.status_code != 200:
                print(f"Não consegui conexão {numero} (HTTP {response.status_code})")
                continue
            
            try:
                resposta = response.json()
                resumo = resposta["choices"][0]["message"]["content"]
            except (ValueError, KeyError, IndexError, TypeError):
                print(f"Resposta inválida da API para {numero}")
                continue
                
            texto_corrigido = " ".join(resumo.split())
            texto_corrigido = texto_corrigido.replace("'", "")
            texto_corrigido = texto_corrigido.replace("‑","-")
            sql_resumo = f"UPDATE anterioridades_desc SET incoerencia='{texto_corrigido}' WHERE numero='{numero}';"
            print(sql_resumo)
            f.write(sql_resumo + "\n")